# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 Mass Spectrometry dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets and their fields using `@id` to reference each entity.


In [ ]:
# Display all record sets with their IDs and associated fields
from pprint import pprint

record_sets = getattr(metadata, 'recordSet', [])

if record_sets:
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f'Record set: {rs_name} (@id={rs_id})')
        for field in getattr(rs, 'field', []):
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            print(f'  Field: {field_name} (@id={field_id}, dataType={getattr(field, "dataType", None)})')
        print()
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# If recordSet is not part of the package, try to auto-list them using the public schema
# and extract by iterating through records and inferring available record sets.

# Get all available record set IDs
record_sets = getattr(metadata, 'recordSet', [])

# Some datasets may use a single main record set, or have more
record_set_ids = []
if record_sets:
    for rs in record_sets:
        if hasattr(rs, '@id'):
            record_set_ids.append(getattr(rs, '@id'))

if not record_set_ids:
    # Fallback: Check if metadata has non-empty hasPart with @type RecordSet
    if hasattr(metadata, 'hasPart'):
        for part in getattr(metadata, 'hasPart', []):
            if getattr(part, '@type', None) == 'RecordSet':
                record_set_ids.append(getattr(part, '@id'))

if not record_set_ids:
    # Try to discover available recordSet IDs by inspecting the Croissant schema
    # This may be accessible via the dataset class, but here we print a warning.
    print('No record sets found. Please check dataset schema.')

if record_set_ids:
    print('Available record set IDs:', record_set_ids)
else:
    print('No record sets available.')

In [ ]:
# For demonstration purposes, let's select the first available record set (if any)
dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading records for record_set {record_set_id} ...")
        try:
            # Load records from the record set
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records. Sample columns: {df.columns.tolist()[:8]}")
        except Exception as e:
            print(f"Error loading {record_set_id}: {e}")

    # Preview the first record set
    main_record_set_id = record_set_ids[0]
    print(f'Columns in main record set ({main_record_set_id}):')
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No dataframes loaded due to absence of record sets.')

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA—such as filtering, normalization, and grouping—using one of the main numeric columns from the chosen record set.

In [ ]:
# Choose a main record set and analyze numeric fields
import numpy as np
from pandas.api.types import is_numeric_dtype

# Pick the main DataFrame
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"Chosen record set: {main_record_set_id}")
    
    # List all numeric columns available
    numeric_cols = [col for col in df.columns if is_numeric_dtype(df[col])]
    print("Numeric fields available:", numeric_cols)
    
    # Select a numeric field (example: Peptide count or Coverage field)
    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].quantile(0.75) if len(df)>10 else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field, e.g. 'description' or similar
        # (if such field exists and has modest cardinality)
        object_cols = [col for col in df.columns if df[col].dtype == 'object']
        for cat_field in object_cols:
            n_uniques = df[cat_field].nunique()
            # Ignore high-cardinality fields
            if 1 < n_uniques < 20:
                group_field = cat_field
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())
                break
    else:
        print('No numeric fields found for EDA.')
else:
    print('EDA not available—no record sets or dataframes.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field from the main record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_cols:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric fields available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load Croissant metadata and data with `mlcroissant`.
- Explore record sets and fields with reference to their `@id`.
- Extract tabular data into DataFrames for analysis.
- Perform EDA, basic filtering, normalization, and grouping on numeric fields.
- Visualize data distributions to gain insight into the dataset structure.

For advanced analysis, you can further investigate more fields or record sets using the Croissant schema's `@id` syntax.
